In [ ]:
import polars as pl
import re


def get_team_name(column_name: str) -> str:
    match = re.search(r"^(.*?)\s*\(gols\)", str(column_name))

    if match:
        return match.group(1).strip()

    raise ValueError(f"Não foi possível extrair o time da coluna: {column_name}")

def create_result_file(input_file: str, output_file: str, sheet_name: str) -> None:
    # =========================================================
    # LEITURA
    # =========================================================

    df = pl.read_excel(
        input_file,
        sheet_name=sheet_name,
    )

    # Primeiras 6 colunas são informacionais
    fixed_cols = df.columns[:6]

    email_col = "E-mail corporativo"

    # Colunas das partidas
    match_cols = df.columns[6:]

    rows = []

    # =========================================================
    # TRANSFORMAÇÃO
    # =========================================================

    for row in df.iter_rows(named=True):

        email = row[email_col]

        # percorre de 2 em 2
        for i in range(0, len(match_cols), 2):

            team_a_col = match_cols[i]
            team_b_col = match_cols[i + 1]

            team_a = get_team_name(team_a_col)
            team_b = get_team_name(team_b_col)

            goals_a = row[team_a_col]
            goals_b = row[team_b_col]

            if goals_a > goals_b:
                result = team_a
            elif goals_b > goals_a:
                result = team_b
            else:
                result = "Empate"

            rows.append(
                {
                    "E-mail_Corporativo": email,
                    "Partida": f"{team_a} x {team_b}",
                    "Resultado": result,
                    "Placar_Time_A": goals_a,
                    "Placar_Time_B": goals_b,
                }
            )

    # =========================================================
    # DATAFRAME FINAL
    # =========================================================

    result_df = pl.DataFrame(rows)

    # =========================================================
    # EXPORTAÇÃO
    # =========================================================

    result_df.write_excel(
        output_file,
        worksheet="Resultado",
    )

    print(f"Arquivo gerado: {output_file}")


Could not determine dtype for column 4, falling back to string


AttributeError: 'NoneType' object has no attribute 'group'

In [ ]:
INPUT_FILE = "C:\\Users\\JoãoCaparroz\\Downloads\\1ª Rodada Copa do Mundo 2026.xlsx"
OUTPUT_FILE = "C:\\Users\\JoãoCaparroz\\Downloads\\1ª Rodada Copa do Mundo 2026 - Resultado.xlsx"
SHEET_NAME = "Base"

create_result_file(INPUT_FILE, OUTPUT_FILE, SHEET_NAME)